In [1]:
!pip install ydata-profiling --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.5/296.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 687.8/687.8 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.8/104.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 79.6 MB/s eta 0:00:00


## Imports

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ydata_profiling import ProfileReport

## Load the dataset

In [3]:
#Loading the dataset with error handling
try:
  df = pd.read_csv('/content/ecommerce_dataset.csv', delimiter=";")
except FileNotFoundError: #Error Handling
  print("Error: 'ecommerce_dataset.csv' not found. Please ensure the file exists and the path is correct.")
except pd.errors.ParserError:
  print("Error: Unable to parse the CSV file. Check the delimiter and file format.")
except Exception as e:
  print(f"An unexpected error occurred: {e}")
else:
  #If DataFrame creation is successful, print first 10 rows
  display(df.head(10))

,Timestamp,User ID,Cross-device user ID,Existing client,Event type,Product ID,Product Price,Product quantity,Product category,Environment,Device type,Browser family,User location
0,1460462764,NaN,NaN,0,Basket,Unknown,11.0,1.0,Medium Items,web,Desktop,safari,Inland
1,1490572501,0c6205543f3f6a7e8082,NaN,0,Listing,Unknown,NaN,NaN,Packages,web,Desktop,edge,Outside
2,1490572501,0c6205543f3f6a7e8082,NaN,0,Listing,Unknown,NaN,NaN,Packages,web,Desktop,edge,Outside
3,1490572501,0c6205543f3f6a7e8082,NaN,0,Listing,Unknown,NaN,NaN,Packages,web,Desktop,edge,Outside
4,1490571486,0cd8953a90d837c045c9,NaN,0,Listing,Unknown,NaN,NaN,Packages,web,iPhone,mobile safari,Inland
5,1490571486,0cd8953a90d837c045c9,NaN,0,Listing,Unknown,NaN,NaN,Packages,web,iPhone,mobile safari,Inland
6,1490571486,0cd8953a90d837c045c9,NaN,0,Listing,Unknown,NaN,NaN,Packages,web,iPhone,mobile safari,Inland
7,1490570571,fbe0702296f94a8d5001,c05f62dea7a788f840d6,0,Product,Unknown,NaN,1.0,Packages,web,Desktop,chrome,Inland
8,1490569249,fbe0702296f94a8d5001,c05f62dea7a788f840d6,0,Listing,Unknown,NaN,NaN,Packages,web,Desktop,chrome,Inland
9,1490569249,fbe0702296f94a8d5001,c05f62dea7a788f840d6,0,Listing,Unknown,NaN,NaN,Packages,web,Desktop,chrome,Inland


## Dataset Profile report

In [4]:
#To find overview of the dataset
profile = ProfileReport(df, title="Dataset Profiling Report")
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

##Cleaning of the dataset

In [5]:
df = df.drop_duplicates() #dropping the duplicates

#Converting the timestamp to data from unix time system
df['Date'] = pd.to_datetime(df['Timestamp'], unit = 's')
df['Date'] = pd.to_datetime(df['Date']).dt.date
col_timestamp_index = df.columns.get_loc('Timestamp')
df.insert(col_timestamp_index + 1, 'Date', df.pop('Date'))#Reordering the dataset for better readability

#Finding the Revenue from Product Price nad Product Quantity
df['Revenue'] = df['Product Price'] * df['Product quantity']
col_product_qty = df.columns.get_loc('Product quantity')
df.insert(col_product_qty + 1, 'Revenue', df.pop('Revenue')) #Reordering the dataset for better readability

#Printing the columns in the dataset
print('The final dataset(cleaned) is: ')
for col in df.columns:
  print('   =>', col)

<ipython-input-5-9801ad4e0758>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = pd.to_datetime(df['Timestamp'], unit = 's')
<ipython-input-5-9801ad4e0758>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = pd.to_datetime(df['Date']).dt.date
<ipython-input-5-9801ad4e0758>:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org

The final dataset(cleaned) is: 
   => Timestamp
   => Date
   => User ID
   => Cross-device user ID
   => Existing client
   => Event type
   => Product ID
   => Product Price
   => Product quantity
   => Revenue
   => Product category
   => Environment
   => Device type
   => Browser family
   => User location


In [6]:
#Plotting the distribution of product prices
fig = px.histogram(df, x="Product Price", nbins=50, title='Distribution of Product Prices of all purchases')
fig.show()

# **1. What is the evolution of the revenue during the year? Can you pinpoint and explain particular events during the year?**

In [35]:
#Keeping the columns that are needed for the analysis
cols_to_keep = ['Date', 'Event type', 'Revenue']
df_revenue = df[cols_to_keep]

#Filtering the data based on Sales in Event type
df_revenue = df_revenue[df_revenue['Event type'] == 'Sales']

#Grouping the dataset by date and adding the all the revenues for individual dates
df_revenue = df_revenue.groupby('Date')['Revenue'].sum()
df_revenue = df_revenue.reset_index()

#Calculating the weekly moving average for better idea about the trend (Since the date-wise is too noisy to understand)
df_revenue['moving_average_by_week'] = df_revenue['Revenue'].rolling(window = 7).mean()
df_revenue.sort_values('Date', inplace = True)

#Printing the first 7 rows of the dataset (Choosing 7 because it will enable us to see the data since we used weekly moving average as metric here)
print(df_revenue.head(7))

         Date  Revenue  moving_average_by_week
0  2016-03-28    871.0                     NaN
1  2016-03-29    533.0                     NaN
2  2016-03-30    682.0                     NaN
3  2016-03-31    431.0                     NaN
4  2016-04-01    282.0                     NaN
5  2016-04-02    330.0                     NaN
6  2016-04-03    678.0              543.857143


In [9]:
#plot the chart
fig = px.line(df_revenue, x='Date',
              y=['Revenue', 'moving_average_by_week'],
              title='Daily Revenue Over Time With Weekly Moving Average',
              line_shape='linear')

#Update the plot for cleaner visualization
fig.data[0].line.color = '#FFA15A'
fig.data[0].name = 'Revenue'
fig.data[1].line.color = '#1F77B4'
fig.data[1].name = 'moving_average_by_week'

fig.update_layout(yaxis_title='Revenue (In Local Currency)',
                 xaxis_title = 'Date')

fig.update_layout(
    title = dict(
        x = 0.5,
        xanchor = 'center'
    ),
    legend=dict(
        x=0.5,
        y=-0.3,
        xanchor='center',
        yanchor='top',
        orientation='h'
    ),
    margin=dict(b=50)  # Margin increased for legend
)
#Show the plot
fig.show()


#Group by Date and sum the revenues, then reset the index
df_grouped = df.groupby('Date')['Revenue'].sum().reset_index()
#Convert 'Date' to datetime
df_grouped['Date'] = pd.to_datetime(df_grouped['Date'])
#Set 'Date' as the DataFrame index for resampling
df_grouped.set_index('Date', inplace=True)
#Resample to get weekly total revenue
df_weekly = df_grouped['Revenue'].resample('W').sum().reset_index()
#Plot the weekly revenue
fig = px.line(df_weekly, x='Date', y='Revenue', title='Total Revenue Over time by Week' )
fig.data[0].line.color = '#D62728'
#Updating the layout for cleaner visualisation
fig.update_layout(
    title=dict(
        x=0.5,
        xanchor='center'
    ),
    yaxis_title='Revenue (In Local Currency)',
    xaxis_title='Week'
)
#Show the plot
fig.show()

From the first plot, it is evident, the noise is daily revenue, so considered weekly moving average to give an overall idea about the trend.

From the above plot, we can easily identify the peaks in revenue and slump in Revenue.

Major Revenue Peak:

=> May 1, 2016 - Labour day/ Mother's Day

=> September 4th, 2016 - Back to school promotions

=> November 27th, 2016 - Halloween, Black Friday, Cyber Monday, Thanksgiving

=> December 11th, December 18th 2016 - Christmas and holiday season (Gifts to people, Secret Santa in work spaces, family and universities)

=> February 12th, 2017 - Valentines day and Chinese New year

Major Revenue Slump:

=> July 17th, 2016 - Summer slump (People tend to go on vacations, so slump in revenue)

=> April, 2017 - Post festival slump (Easter, post spring promotional offers)

## Additional revenue onsite recommendations

### Revenue by Time


Exploring how the Revenue fluctuation during different times of the day or week

In [10]:
#Selecting the rows that is needed for the above analysis
cols_to_keep_1 = ['Timestamp', 'Event type', 'Revenue', 'User location']
df_revenue_time = df[cols_to_keep_1]

#Converting unix time to UTC
df_revenue_time['Timestamp'] = pd.to_datetime(df_revenue_time['Timestamp'], unit='s')

#Extracting the hour from timestamp
df_revenue_time['Hour'] = df_revenue_time['Timestamp'].dt.hour
df_revenue_time['Day of the Week'] = df_revenue_time['Timestamp'].dt.day_name()

<ipython-input-10-f318052225dd>:6: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

<ipython-input-10-f318052225dd>:9: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

<ipython-input-10-f318052225dd>:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [11]:
#Filter inland and outland data
df_revenue_time_inland = df_revenue_time[df_revenue_time['User location'] == 'Inland']
df_revenue_time_outland = df_revenue_time[df_revenue_time['User location'] == 'Outside']

#Group revenue by hour and day of the week for inland and outland
revenue_by_hour_inland = df_revenue_time_inland.groupby(['Day of the Week', 'Hour']).agg({'Revenue': 'sum'}).reset_index()
revenue_by_hour_outland = df_revenue_time_outland.groupby(['Day of the Week', 'Hour']).agg({'Revenue': 'sum'}).reset_index()

#Calculate average revenue by hour for both inland and outland
avg_revenue_by_hour_inland = revenue_by_hour_inland['Revenue'].mean()
avg_revenue_by_hour_outland = revenue_by_hour_outland['Revenue'].mean()

#Pivot tables for inland and outland, set proper day ordering
revenue_matrix_inland = revenue_by_hour_inland.pivot(index='Hour', columns='Day of the Week', values='Revenue').fillna(0)
revenue_matrix_outland = revenue_by_hour_outland.pivot(index='Hour', columns='Day of the Week', values='Revenue').fillna(0)

#Set the  order for the Day of the Week
days_of_week_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

#Reindex columns to follow the correct order
revenue_matrix_inland = revenue_matrix_inland[days_of_week_order]
revenue_matrix_outland = revenue_matrix_outland[days_of_week_order]

#Create interactive heatmap for inland users using average revenue per hour/day combination
fig_inland = px.imshow(
    revenue_matrix_inland,
    color_continuous_scale='oranges',  # Using Brewer Blues color scale
    labels={'x': 'Day of the Week', 'y': 'Hour of Day', 'color': 'Revenue (in local currency)'},
    title='Revenue Heatmap by Hour and Day of the Week (Inland)',
    aspect="auto"  # Ensures correct aspect ratio for better readability
)

#Fix the X-axis and Y-axis labels
fig_inland.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(7)),
        ticktext=days_of_week_order  # Ensuring days are ordered correctly
    ),
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(24)),
        ticktext=[f'{i}' for i in range(24)],  # Hour labels
    ),
    title_x=0.5  # Center the title
)

#Print the average revenue for inland
print("\n The average revenue by hour in inland is:", round(avg_revenue_by_hour_inland, 2))

#Show the plot
fig_inland.show()

#Create interactive heatmap for outland users using average revenue per hour/day combination
fig_outland = px.imshow(
    revenue_matrix_outland,
    color_continuous_scale='oranges',  # Using Brewer Blues color scale
    labels={'x': 'Day of the Week', 'y': 'Hour of Day', 'color': 'Revenue (in local currency)'},
    title='Revenue Heatmap by Hour and Day of the Week (Abroad)',
    aspect="auto"
)

#Fix the X-axis and Y-axis labels for outland
fig_outland.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(7)),
        ticktext=days_of_week_order  # Ensuring days are ordered correctly
    ),
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(24)),
        ticktext=[f'{i}' for i in range(24)],  # Hour labels
    ),
    title_x=0.5  # Center the title
)

#Print the average revenue for outland
print("\n The average revenue by hour out of Abroad is:", round(avg_revenue_by_hour_outland, 2))

#Show the plot
fig_outland.show()




 The average revenue by hour in inland is: 29101.05



 The average revenue by hour out of Abroad is: 4104.88


**Inference:**

**Best Performing Days & Hours:**

**Inland Inference:**

Sunday ( 3PM to 8PM)

Monday ( 7PM to 8PM)

**Abroad Inference:**

Tuesday (1PM)

Sunday (2PM to 8PM)


**Low Performing Days & Hours:**

**Inland Inference:**

Friday & Saturday (Relatively low)

**Abroad Inference:**

Friday & Saturday (Relatively low)

**Metric:**

**Inland :** Average Revenue by Hour : 29,101

**Abroad :** Average Revenue by Hour : 4,105

**Recommendations:**

1. Optimize Campaign for peak hours
2. Target High Engagement periods
3. Dynamic Pricing strategy
4. Improive customer rtetnion for low performing hours




### Revenue by Location

In [12]:
#Filtering the data by considering the Sales and selecting the relevant columns for analysis
df_region = df[df['Event type'] == 'Sales'][['Date', 'User location', 'Revenue']]

#Ensuring the date is in date time format
df_region['Date'] = pd.to_datetime(df_region['Date'])

#Grouping the data by Date and User location and summing the revenue
df_region = df_region.groupby(['Date', 'User location']).sum()

#Converting the dataset to pivot table
df_region = df_region.pivot_table(index='Date', columns='User location', values='Revenue', aggfunc='sum')

#Resampling the data by weekly and resetting the index for clean dataframe
df_region = df_region.resample('W').sum().reset_index()

In [13]:
#Plotting the revenue over time split by user location using Line plot

fig = px.line(df_region,
              x = 'Date',
              y=['Inland', 'Outside', 'Unknown'],
              title = "Revenue over time split by User location")

#Update the plot for cleaner visualizations
fig.update_layout(
    title = dict(
        x = 0.5,
        xanchor = 'center'
    ),
    legend = dict(
        x = 0.5,
        y = -0.3,
        xanchor = 'center',
        yanchor = 'top',
        orientation = 'h'
    ),
    xaxis_title = 'Week',
    yaxis_title = 'Revenue (In Local Currency)'
)
#Show the plot
fig.show()

**Inference:**

From the plot it is evident that:

-  Inland users contribute more towards the revenue. (Primary Source)

- There is a sudden spike in the Unknown category after the android app launched, in this case we need to have a proper tracking of the location of the users who purchase through the app, so that personalized recommendations can be made for those users

-   The holday season in December sees a spike in the outside users, a possible reason that they might gift to their friends and family who are in inland.

**Recommendations:**

- Need to improve the data tracking of app users.

- Exploring the International markets targetting similar customer profiles of that of Inland users and tailoring the offerings based on the cultural preferences of specific international market

### Customer Segmentation based on Purchase Behavior (RFM Analysis for Mid or Short term Revenue Improvement)

In [14]:
#Filtering the sales data from original dataset
df_sales = df[df['Event type'] == 'Sales']

#Finding the recent/latest date
recent_date = df_sales['Date'].max()

#Ensuring it in proper format
recent_date = pd.to_datetime(recent_date)

#Grouping the sales data by User ID and Date
recency = df_sales.groupby('User ID')['Date'].max().reset_index()
recency['Date'] = pd.to_datetime(recency['Date'])
recency['Recency'] = (recent_date - recency['Date']).dt.days

#Grouping the sales data by User ID and counting the occurences for frequency factor
frequency = df_sales.groupby('User ID').size().reset_index(name = 'Frequency') # considering for the year (the dataset is for a year, counting the repeat customers for the available dataset)

#Grouping to sales daya by User ID and Revenue for monetary factor
monetary = df_sales.groupby('User ID')['Revenue'].sum().reset_index()
monetary.columns = ['User ID', 'Monetary']

#Merging recency, frequency and Monetary
rfm = recency.merge(frequency, on = 'User ID').merge(monetary, on = 'User ID')

#Assigning the scores to recency, frequency and monetary based on quantiles
rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1])
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5], duplicates='drop')
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5], duplicates='drop')

#Calculating the RFM score
rfm["RFM_Score"] = rfm["R_Score"].astype(int) + rfm["F_Score"].astype(int) + rfm["M_Score"].astype(int)

#Setting the threshold classify customers based on Monetary
high_customer_threshold = rfm['Monetary'].quantile(0.75)
low_customer_threshold = rfm['Monetary'].quantile(0.25)

#Assigning customer segement based on Monetary
rfm["Segment"] = rfm["Monetary"].apply(
    lambda x: "High-Value Customer" if x >= high_customer_threshold else
              "Low-Value Customer" if x <= low_customer_threshold else
              "Mid-Tier Customer"
)


In [15]:
#Function to classify the customer based on RFM score
def classify_customer(rfm_score):
    rfm_score = int(rfm_score)  #Confirming the score is integer

    if rfm_score >= 12:
        return "Best Customers"
    elif 0 <= rfm_score < 6:
        return "At Risk Customers"
    else:
        return "General Customers"

rfm['Segment_rfm'] = rfm['RFM_Score'].apply(classify_customer)

In [16]:
segment_counts = rfm["Segment_rfm"].value_counts()

#Plotting the graph
fig = px.bar(segment_counts,
             x = segment_counts.index,
             y = segment_counts.values,
             title = "Customer Segmentation based on RFM analysis",
             labels = {"x" : "Customer Segmentation", "y" : "Count"},
             color = segment_counts.index)
fig.update_traces(
    marker = dict(
        line = dict(
        color = 'black', width = 1)
    )
)

fig.update_layout(
    title = dict(
        x = 0.5,
        xanchor = 'center'
    ),
    legend = dict(
        x = 0.5,
        y = -0.3,
        xanchor = 'center',
        yanchor = 'top',
        orientation = 'h'
    )
)
#Show the plot
fig.show()


**Inference:**

From the bar graph, it is evident that there are lot of general customers consitute the major part of the customer base. Meanwhile, the best customers, who buy frequently, spend a lot and made a recent purchase consistute a smaller chunk of the total customers. Strategic decision based on the individual customer group need to addressed for maximising the revenue (in medium/ short term)

**Recommendations:**
**1. Loyal Customers:**

Loyalty programs & Exclusive offers, Upselling & cross selling, Personalized offers, Limited time offers and Referral programs

**2. At Risk Customers(Low recency & Frequency):**

Reactivation or Win back campaigns, Engagement emails, Feedback surveys

**3. General Customers:**

Product bundling, Retargeting Ads, Free Shipping thresholds, Discounts, Product recommendations

In [17]:
#Pie-Chart based on the Customer Profitability Segmentation
profitability_counts = rfm["Segment"].value_counts()

fig1 = px.pie(values=profitability_counts,
              names=profitability_counts.index,
              title="Customer Profitability Segmentation (Based on Monetary)",
              hole=0.3)
fig1.update_traces(
    marker = dict(
        line = dict(
        color = 'black', width = 1)
    )
)
fig1.update_layout(
    title = dict(
        x = 0.5,
        xanchor = 'center'
    ),
    legend = dict(
        x = 0.5,
        y = -0.3,
        xanchor = 'center',
        yanchor = 'top',
        orientation = 'h'
    )
)
#Show the plot
fig1.show()


**Inference:**

From the pie chart, it is evident that around ~26% consitutes low value customers, so strategic decision has to be made keeping in mind to nurture the high value customers but also aim to convert the Low and Mid tier customers to High value customers.

**Recommendations:**

1. Increase Purchase Frequency
2. Increase Average order value
3. Improve sutomer engagement & Retention

### Customer Journey Analysis

In [18]:
#Define journey steps and event mapping
journey_steps = ["Listing", "Product", "Basket", "Sales"]
event_mapping = {
    "listing": "Listing",
    "product": "Product",
    "basket": "Basket",
    "sales": "Sales"
}

#Function to calculate customer journey and drop-off rates for a specific device type
def customer_journey_by_device(df_device):
    # Categorize events into journey stages
    df_device["Journey_Stage"] = df_device["Event type"].str.lower().map(event_mapping)

    #Count unique users at each stage
    journey_counts = df_device.groupby("Journey_Stage")["User ID"].nunique().reindex(journey_steps)

    #Calculate drop-off rates
    journey_counts = journey_counts.dropna().reset_index()
    journey_counts.columns = ["Journey_Stage", "Unique_Users"]
    journey_counts["Dropoff_Rate (%)"] = journey_counts["Unique_Users"].pct_change().fillna(0) * -100

    #Calculate drop-off percentage at each stage
    journey_counts["Dropoff_Absolute"] = journey_counts["Unique_Users"].diff().fillna(0) * -1
    journey_counts["Dropoff_Percentage"] = (journey_counts["Dropoff_Absolute"] / journey_counts["Unique_Users"].shift(1)) * 100
    journey_counts["Dropoff_Percentage"] = journey_counts["Dropoff_Percentage"].fillna(0)

    #Calculate conversion rate to the next stage
    journey_counts["Conversion_Rate (%)"] = (journey_counts["Unique_Users"] / journey_counts["Unique_Users"].shift(1)) * 100
    journey_counts["Conversion_Rate (%)"] = journey_counts["Conversion_Rate (%)"].fillna(100)  # First stage is 100%

    #Identify the biggest drop-off stage
    biggest_dropoff_stage = journey_counts.loc[journey_counts["Dropoff_Percentage"].idxmax()]

    return journey_counts, biggest_dropoff_stage

#Filter data for Web users
df_web = df[df['Environment'] == 'web']
journey_counts_web, biggest_dropoff_web = customer_journey_by_device(df_web)

#Plotting for Web
fig_web = px.bar(
    journey_counts_web,
    x="Journey_Stage",
    y="Unique_Users",
    title="Customer Journey Analysis for Web",
    labels={"Journey_Stage": "Customer Journey Stage", "Unique_Users": "Number of Unique Users"},
    color="Journey_Stage",
    text=journey_counts_web["Unique_Users"]
)

fig_web.update_layout(
    title=dict(x=0.5, xanchor='center'),
    legend=dict(x=0.5, y=-0.3, xanchor='center', yanchor='top', orientation='h')
)

#Filter data for Android apps
df_app = df[df['Environment'] != 'web']
journey_counts_app, biggest_dropoff_app = customer_journey_by_device(df_app)

#Plotting for Android apps
fig_app = px.bar(
    journey_counts_app,
    x="Journey_Stage",
    y="Unique_Users",
    title="Customer Journey Analysis for Android apps",
    labels={"Journey_Stage": "Customer Journey Stage", "Unique_Users": "Number of Unique Users"},
    color="Journey_Stage",
    text=journey_counts_app["Unique_Users"]
)

fig_app.update_layout(
    title=dict(x=0.5, xanchor='center'),
    legend=dict(x=0.5, y=-0.3, xanchor='center', yanchor='top', orientation='h')
)

#Display the graphs
fig_web.show()
fig_app.show()

#Displaying the drop-off stage for both Desktop and Other Devices
print("Biggest drop-off stage for Web:\n", biggest_dropoff_web)
print("Biggest drop-off stage for Android:\n", biggest_dropoff_app)

<ipython-input-18-f9a6e7308fdb>:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

<ipython-input-18-f9a6e7308fdb>:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Biggest drop-off stage for Web:
 Journey_Stage             Basket
Unique_Users               18493
Dropoff_Rate (%)       80.366072
Dropoff_Absolute         75696.0
Dropoff_Percentage     80.366072
Conversion_Rate (%)    19.633928
Name: 2, dtype: object
Biggest drop-off stage for Android:
 Journey_Stage              Sales
Unique_Users              3245.0
Dropoff_Rate (%)       80.905025
Dropoff_Absolute         13749.0
Dropoff_Percentage     80.905025
Conversion_Rate (%)    19.094975
Name: 1, dtype: object


**Inference:**

From the graph it is evident there is a huge drop out from the Product to Basket, this consititutes around ~80% and only ~20% who has viewed the product added it to the cart. The reason could be users are interested in product but reluctance to commit. Since the app does not have the listing and basket features, we have only product to sales.

**Possible Causes:**

1. Unexpected costs like shipping, taxes, fees
2. Lack of payment options
3. Delivery time concerns
4. Trust issues about the seller

**Recommendations:**

1. Retargeting or Cart reminder emails
2. Pop up with an incentive or discounts when the user tries to leave the page
3. Easier Checkout process, could be improved by A/B Testing
4. Progress Indicators during Checkout


In [19]:
display(journey_counts_web)
display(journey_counts_app)

,Journey_Stage,Unique_Users,Dropoff_Rate (%),Dropoff_Absolute,Dropoff_Percentage,Conversion_Rate (%)
0,Listing,3096,-0.000000,-0.0,0.000000,100.000000
1,Product,94189,-2942.280362,-91093.0,-2942.280362,3042.280362
2,Basket,18493,80.366072,75696.0,80.366072,19.633928
3,Sales,12343,33.255827,6150.0,33.255827,66.744173


,Journey_Stage,Unique_Users,Dropoff_Rate (%),Dropoff_Absolute,Dropoff_Percentage,Conversion_Rate (%)
0,Product,16994.0,-0.000000,-0.0,0.000000,100.000000
1,Sales,3245.0,80.905025,13749.0,80.905025,19.094975


### Revenue by Product Category

In [20]:
#Grouping the sales data and product category
revenue_by_category = df_sales.groupby(['Date', 'Product category'])['Revenue'].sum().reset_index()

#Creating the pivot table
pivot_revenue = revenue_by_category.pivot_table(index = 'Date', columns = 'Product category', values = 'Revenue', aggfunc = 'sum').reset_index()

#Converting the Date to proper format and setting it as index
pivot_revenue ['Date'] = pd.to_datetime(pivot_revenue['Date'])
pivot_revenue.set_index('Date', inplace = True)

#Resampling the data to weekly frequency
weekly_revenue = pivot_revenue.resample('W').sum().reset_index()

#Plotting the graph
fig_revenue = px.line(weekly_revenue, x='Date', y=['Large Items', 'Medium Items', 'Packages', 'Small Items'],
                      title = "Revenue over Time split by Product Category")
fig_revenue.update_layout(
    legend = dict( x=0.5, y = -0.3, xanchor ='center', yanchor = 'top', orientation = 'h'),
    margin = dict(b=100)
)
fig_revenue.update_yaxes(title_text = 'Revenue')
fig_revenue.show()

#Grouping the sales by Product category with aggregrating Revenue and Quantity
category_stats = df_sales.groupby('Product category').agg(
    total_revenue = ('Revenue', 'sum'),
    total_quantity = ('Product quantity', 'sum')
).reset_index()

#Calculating average selling price
category_stats['average_selling_price'] = category_stats['total_revenue'] / category_stats['total_quantity']

#Sorting the categories in descending order
category_stats_sorted = category_stats.sort_values(by='average_selling_price', ascending=False)

#Display the sorted category table
display(category_stats_sorted)

,Product category,total_revenue,total_quantity,average_selling_price
2,Packages,222471.0,8135.0,27.347388
0,Large Items,37143.0,1668.0,22.267986
1,Medium Items,83288.0,5322.0,15.649756
3,Small Items,124900.0,8347.0,14.963460


**Inference:**

1. Large Items has sold comparatively less than the other product categories
2. Packages have peaked during all the seasonal time period, indicating users purchase trend during that time period
3. Packages dominate the other product categories in terms of revenue and average selling price, contributing significant chunk to revenue.
4. Even during the dip in Revenue on July 17th, the packages and large items seems to maintain the similar trend. So people avoid purchasing the medium and small items, which needs further investigation
5. Small items has significant potential to increase revenue, but their average selling price does not support the cause.

**Recommendations:**

1. Analyze New Customer Purchasing Behaviour
2. Price Optimisation
3. Focus on July 17th Dip
4. Targeted Campaigns


# 2. Over the year, what devices have shoppers used to browse this merchant's site? Explain the main changes.

In [21]:
#Selecting the columns as per the problem statement
df_device = df[['Date', 'Device type']]

#Aggregrating the data using size and converting it into pivot table along with handling missing values
df_device_agg = df_device.groupby(['Date', 'Device type']).size().unstack(fill_value=0)

#Converting index to datetime
df_device_agg.index = pd.to_datetime(df_device_agg.index)

#Resampling the data by week
df_device_agg = df_device_agg.resample('W').sum()

#Line plot for events recorded over time for each device type
fig = px.line(df_device_agg, x = df_device_agg.index, y = ['Android - Smartphone', 'Android - Tablet', 'Desktop',
                                                   'Mobile - Other', 'Unknown', 'iPad', 'iPhone'], title = 'Events Recorded over time by Different Devices')

#Updating the layout for cleaner visualisation
fig.update_layout(
    title = dict(
        x = 0.5,
        xanchor = 'center'
    ),
    legend=dict(
        x=0.5,
        y=-0.3,
        xanchor='center',
        yanchor='top',
        orientation='h'
    )
)
fig.update_yaxes(title_text = "Events Recorded")
fig.update_xaxes(title_text = "Month")
fig.show()

**Inference:**

From the graph, it is evident that there are more desktop sessions than the other sessions (Android, iOS and other) combined. But we can see the number of users have increased from August 28th, potentially leading to new way of accessing the merchant sites (With the subsequent question, it is very evident that it is the Launch of the android app), because the accessing the merchant site through Android-Smartphone and Android-Tablet takes a dip post August 28th, confirming the assumption.

**Recommendations:**

1. If there is not any specific app designed for iOS, launching an app for iOS should be highly beneficial. This will help in tapping the high engagement iOS userbase, improving in usage of the merchant site.

2. Requires further study because, the Iphone events took a decline after launching the android app (particularly from the week leading to October 30th) because it might be helpful in user retention.

# 3. In your opinion, do Applications perform better than Mobile Web browsing?

In [22]:
#Selecting the columns as per the problem statement
df_env = df[['Date','User location','Browser family','Device type','Environment','Event type']]

#Filtering the data, grouping it by Date and Environment, counting the occurences and converting it pivot table
df_environment = df_env[df_env['Device type'] != 'Desktop'] \
    .groupby(['Date', 'Environment'], as_index=False) \
    .size() \
    .pivot_table(index='Date', columns='Environment', values='size', aggfunc='sum')

#Converting the date as index
df_environment.index = pd.to_datetime(df_environment.index)

#Resampling the data by week
df_environment = df_environment.resample('W').sum()

#Line plot for the events recorded over time to the type of environment
fig = px.line(df_environment, x=df_environment.index, y=['app_android', 'web'], title = 'Events Recorded Over Time From App and Web Browsing')

#Update plot layout and legend for cleaner visualisation
fig.update_layout(
    title = dict(
        x = 0.5,
        xanchor = 'center'
    ),
    legend=dict(
        x=0.5,
        y=-0.3,
        xanchor='center',
        yanchor='top',
        orientation='h'
    )
)
fig.update_yaxes(title_text="Events Recorded")
fig.update_xaxes(title_text="Month")
fig.show()


In [23]:
#Filter the dataset (df) for the relevant criteria
df_browser = df[
    (df['Device type'] != 'Desktop') &
    (df['Date'] >= pd.to_datetime('2016-08-28').date())
][['Date', 'Device type', 'Environment', 'Revenue', 'User ID', 'Existing client', 'Event type', 'Product Price', 'Product quantity', 'Product category']]

#Count unique users by environment
user_counts = df_browser.groupby('Environment')['User ID'].nunique()

#Calculate revenue generated by environment for 'Sales' events
generated_revenue = df_browser[df_browser['Event type'] == 'Sales'].groupby('Environment')['Revenue'].sum()

#Print user counts and revenue by environment
print('\nNumber of Users by Environment:')
print(user_counts)
print('\nRevenue Generated by Environment:')
print(generated_revenue)


Number of Users by Environment:
Environment
app_android    17036
web            14284
Name: User ID, dtype: int64

Revenue Generated by Environment:
Environment
app_android    84918.0
web            12951.0
Name: Revenue, dtype: float64


Web Data Analysis

In [24]:
#Filter for web environment
df_browsing_web = df_browser[df_browser['Environment'] == 'web']

#Group by date, User ID, and Event type to get event counts for web
web_event_presence = df_browsing_web.groupby(['Date', 'User ID', 'Event type']).size().unstack(fill_value=0).reset_index()

#Print the counts of various events for the web environment
web_event_presence_listing_events = web_event_presence.Listing.sum()
web_event_presence_basket_events = web_event_presence.Basket.sum()
web_event_presence_product_events = web_event_presence.Product.sum()
web_event_presence_sales_events = web_event_presence.Sales.sum()

#Print the event counts for web
print('web_event_presence_listing_events:', web_event_presence_listing_events)
print('web_event_presence_basket_events:', web_event_presence_basket_events)
print('web_event_presence_product_events:', web_event_presence_product_events)
print('web_event_presence_sales_events:', web_event_presence_sales_events)

#Calculate conversion rates for web
web_product_to_sales_rate = round((web_event_presence_sales_events / web_event_presence_product_events) * 100, 2)
web_basket_to_sales_rate = round((web_event_presence_sales_events / web_event_presence_basket_events) * 100, 2)
web_product_to_basket_rate = round((web_event_presence_basket_events / web_event_presence_product_events) * 100, 2)

#Print the conversion rates for web
print('web_product_to_sales_rate:', web_product_to_sales_rate)
print('web_basket_to_sales_rate:', web_basket_to_sales_rate)
print('web_product_to_basket_rate:', web_product_to_basket_rate)

web_event_presence_listing_events: 2503
web_event_presence_basket_events: 3259
web_event_presence_product_events: 22123
web_event_presence_sales_events: 723
web_product_to_sales_rate: 3.27
web_basket_to_sales_rate: 22.18
web_product_to_basket_rate: 14.73


App Data Analysis

In [25]:
#Filter for app environment
df_browsing_app = df_browser[df_browser['Environment'] == 'app_android']

#Group by date, User ID, and Event type to get event counts for app
app_event_presence = df_browsing_app.groupby(['Date', 'User ID', 'Event type']).size().unstack(fill_value=0).reset_index()

#Print the counts of various events for the app environment
app_event_presence_product_events = app_event_presence.Product.sum()
app_event_presence_sales_events = app_event_presence.Sales.sum()

#Print the event counts for app
print('app_event_presence_product_events:', app_event_presence_product_events)
print('app_event_presence_sales_events:', app_event_presence_sales_events)

#Calculate conversion rates for app
app_product_to_sales_rate = round((app_event_presence_sales_events / app_event_presence_product_events) * 100, 2)

#Print the conversion rate for app
print('app_product_to_sales_rate:', app_product_to_sales_rate)

app_event_presence_product_events: 36441
app_event_presence_sales_events: 4840
app_product_to_sales_rate: 13.28


Conversion Lift Between App and Web

In [26]:
#Calculate conversion lift between app and web for product-to-sales rates
app_to_web_conv_lift = round(((app_product_to_sales_rate - web_product_to_sales_rate) / web_product_to_sales_rate) * 100, 2)

#Print the conversion lift
print('app_to_web_conv_lift:', app_to_web_conv_lift)

app_to_web_conv_lift: 306.12


From the data above, it is evident that the number of app users exceed the web users by ~19%, which directly impacted the revenue generated also. This had a effect of ~555% increase in revenue through android apps. Also, the number of users in app (post the launch of android app) the user base tends to shift their usage to app. The number of app users continues to dominate the web in terms of numbers.  But post March 19th, 2017 there seems to decline in the app users and rise in web usage. This might be because of either of these issue:

=> UX/UI issues

=> App updates or Issues

=> Compatibility issues due to core software upgrade (Eg: Android version change)

=> Improved Web Experience

**Recommendations:**

1. Focus on hybrid strategy, since desktop is dominant device, acquistion of the customers through that and converting them to use the apps where the conversion metrics are muh higher than the mobile web browsing.

2. Launch of iOS app, if it is not available already

3. As of now, the comparisons are between mobile web browsing and app, so extend that to Desktop and verify the metrics to identify the areas of improvement in the respective way of using the merhcant site

4. For all the app interactions, listing and basket is missing. If that is a tracking issue, need to explore that, or incase if that feature is not available in the app, need to integrate that features in app to track usage and drop between the event helps us find better insights of usage.

# 4. Customer Lifetime Value (Long term Revenue Prediction)

In [27]:
import numpy as np
import pandas as pd

#Filter sales data
df_sales = df[df['Event type'] == 'Sales']

#Calculate total revenue per user
customer_revenue = df_sales.groupby('User ID')['Revenue'].sum().reset_index()
customer_revenue.columns = ['User ID', 'Total Revenue']

#Calculate total orders (purchases) per user
customer_orders = df_sales.groupby('User ID')["Product ID"].count().reset_index()
customer_orders.columns = ['User ID', 'Total Orders']

#Merge revenue and orders data into customer_data
customer_data = pd.merge(customer_revenue, customer_orders, on='User ID')

#Calculate Average Purchase Value (APV) per customer
customer_data['Average_PV'] = customer_data['Total Revenue'] / customer_data['Total Orders']

#Total number of customers
total_customers = customer_data['User ID'].nunique()

#Convert 'Date' column to datetime
df_sales['Date'] = pd.to_datetime(df_sales['Date'])

#Calculate customer purchase frequency and purchase date difference
customer_frequency = df_sales.groupby('User ID').agg(
    first_purchase=('Date', 'min'),
    last_purchase=('Date', 'max'),
    total_purchases=('Product ID', 'count')
).reset_index()

#Get the number of purchases per day for each user
user_purchase_count_per_day = df_sales.groupby(['User ID', 'Date']).size().reset_index(name='purchase_count')

#Calculate the max number of purchases on a single day for each user
max_purchase_per_day = user_purchase_count_per_day.groupby('User ID')['purchase_count'].max().reset_index()

#Merge the max purchase per day data into the customer_frequency DataFrame
customer_frequency = pd.merge(customer_frequency, max_purchase_per_day, on='User ID', how='left')

#Calculate the purchase frequency for users with more than one purchase
customer_frequency['purchase_frequency'] = np.where(
    customer_frequency['total_purchases'] > 1,
    (customer_frequency['last_purchase'] - customer_frequency['first_purchase']).dt.days / (customer_frequency['total_purchases'] - 1),
    np.nan  # Set to NaN for one-time users
)

#Adjust purchase frequency to be the maximum of the calculated frequency or the max purchases in a day
customer_frequency['purchase_frequency'] = customer_frequency[['purchase_frequency', 'purchase_count']].max(axis=1)

#Merge customer data with purchase frequency and new columns (first, last, and date difference)
customer_data = pd.merge(customer_data, customer_frequency[['User ID', 'first_purchase', 'last_purchase', 'purchase_frequency']], on='User ID')

<ipython-input-27-06315288c6cb>:25: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [29]:
#Ensuring the datetime format
df_sales['Date'] = pd.to_datetime(df_sales['Date'])

#Define the date range
start_date = pd.to_datetime('2016-04-03')
split_date = pd.to_datetime('2016-10-02')
end_date = pd.to_datetime('2017-04-02')

#Splitting the dataset
df_sales_first_half = df_sales[(df_sales['Date'] >= start_date) & (df_sales['Date'] <= split_date)]
df_sales_second_half = df_sales[(df_sales['Date'] > split_date) & (df_sales['Date'] <= end_date)]

#Calculating the users total purchase in the first half
customer_frequency_first_half = df_sales_first_half[df_sales_first_half['Existing client'] != 1]
customer_frequency_first_half = df_sales_first_half.groupby('User ID')['Product ID'].count().reset_index()
customer_frequency_first_half.columns = ['User ID', 'total_purchases_first_half']

#Calculate the users who made only one purchase
users_with_first_purchase_first_half = customer_frequency_first_half[customer_frequency_first_half['total_purchases_first_half'] == 1]

#Finding the users who made only one purhcase in the first half
users_first_half = users_with_first_purchase_first_half['User ID'].tolist()

#Finding the users who made a purchase in second half
df_second_half_filtered = df_sales_second_half[df_sales_second_half['User ID'].isin(users_first_half)]

#Calculate the number of users from the first half who made purchase in the second half
users_with_second_purchase_in_second_half_list = df_sales_second_half['User ID'].unique().tolist()

#Finding the users who made atleast one purchase in second half
converted_user_list = [user for user in users_with_second_purchase_in_second_half_list if user in users_first_half]

#Finding the probability estimate
probability_of_conversion = len(converted_user_list)/len(users_first_half)

print(probability_of_conversion)

<ipython-input-29-c7f5bbc1baf8>:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



0.06222631942843973


In [30]:
  #Customer lifespan assumption (Customer who is going to maintain the relationship with the business in years)
customer_lifespan = 2

#CLV Calculation for customers with at least one purchase
customer_data['CLV'] = customer_data['Average_PV'] * customer_data['purchase_frequency'] * customer_lifespan

#For users who have made only 1 purchase, estimate their CLV based on the probability of a second purchase
users_1_purchase = customer_data[customer_data['Total Orders'] == 1]

#Calculate the average CLV for users with 2 purchases
users_2_purchases = customer_data[customer_data['Total Orders'] == 2]
avg_clv_2_purchases = users_2_purchases['CLV'].mean()
print("Average CLV for users with 2 purchases is ", avg_clv_2_purchases)

customer_data.loc[customer_data['Total Orders'] == 1, 'CLV'] = probability_of_conversion * avg_clv_2_purchases

#Display the updated customer data with the first purchase, last purchase, their difference, and CLV for one-time users
display(customer_data.head(4))

Average CLV for users with 2 purchases is  638.3509025270758


,User ID,Total Revenue,Total Orders,Average_PV,first_purchase,last_purchase,purchase_frequency,CLV
0,0002eb9e66b0161041a1,13.0,1,13.000000,2016-09-13,2016-09-13,1.0,39.722227
1,00054695e039e7ff1df7,23.0,1,23.000000,2017-01-11,2017-01-11,1.0,39.722227
2,0006640804e2aad43f10,74.0,3,24.666667,2016-12-01,2016-12-01,3.0,148.000000
3,0007321bb96031e6fde5,23.0,1,23.000000,2017-01-15,2017-01-15,1.0,39.722227


In [31]:
#Plotting the CLV in histograms
fig1 = px.histogram(customer_data, x = 'CLV', nbins = 100,
                    title = 'Distribution of Customer Lifetime Value (CLV)',
                    labels = {"CLV" : "Log of Customer Lifetime Value (CLV)"}
                    )

fig1.update_traces(
    marker = dict(
        line = dict(
        color = 'black', width = 1)
    )
)

fig1.update_layout(
    title = dict(
        x = 0.5,
        xanchor = 'center'
    ),
    legend = dict(
        x = 0.5,
        y = -0.3,
        xanchor = 'center',
        yanchor = 'top',
        orientation = 'h'
    ),
    xaxis_title = 'CLV',
    yaxis_title = 'Number of Customers',
    showlegend = False,
)
fig1.update_traces(marker=dict(color='#EB663B'))

fig1.show()

In [32]:
fig_violin = px.violin(customer_data, y='CLV', box=True, points="all",
                       title='Violin Plot of Customer Lifetime Value (CLV)',
                       labels={'CLV': 'Customer Lifetime Value (CLV)'})

# Customizing the layout
fig_violin.update_layout(
    yaxis_title='Customer Lifetime Value (CLV)',
    xaxis_title='Customer',
)

fig_violin.update_traces(marker=dict(color='#EB663B'))

# Show the plot
fig_violin.show()


**Inference:**

From the plots above, it is evident that most of the customers lie around the CLV of 39.

**Assumptions:**


1. Considering the probability that the one time customer will make another purchase, based on the frequency of one and two time purhcases made and it is multiplied with average CLV of users who have made two purchases), these values mostly correspond to the users who have one time purchases. (Probability Estimate)
2. Users who have ordered multiple orders on the same day, have their purchase frequency as same as thier number of orders.
3. If users have a purchase frequency and also they made multiple orders on some other day, then the max among them will be considered

**Recommendations:**

=> Should Focus on Customer Onboarding (Discounts or Exclusive first-time offers, product tutorials and chat support)

=> Loyalty & Reward programs to encourage repeat purchases ( Point based rewards, Referral programs)

=> Re-engagement campaigns for the very low CLV customers

=> Nurture the high CLV customers by providing them VIP treatment ( Personalized Recommendations, Early access to Sale, Birthday Rewards)

=> Should focus more on upselling and cross-selling strategies (Based on their purchase history, personalized recommendations, bundling the products and services)